# IBM SkillsBuild Cyber Security BootCamp 2026 | BharatCares
## Project: Automated Web Application Security & Header Audit Scanner
**Student Name:** Joydip Maiti  
**Track:** Application Security & Vulnerability Assessment  
**Standards:** OWASP Top 10, CWE-693 (Protection Mechanism Failure)  
---
### 🌐 Executive Overview
Misconfigured web application headers and unhardened HTTP parameters represent one of the most common pathways for initial compromise. This project provides an automated, Python-powered security scanner that audits live web applications for missing defensive headers (HSTS, CSP, X-Frame-Options), Cross-Origin Resource Sharing (CORS) flaws, cookie flags (Secure, HttpOnly, SameSite), and SSL/TLS cipher suites.

In [1]:
# Step 1: Import Core Audit Libraries
import os
import requests
import ssl
import socket
from datetime import datetime, timezone
from urllib.parse import urlparse
import json

requests.packages.urllib3.disable_warnings()
print('[+] Security scanner modules initialized.')

### 🛡️ Step 2: Defining OWASP Security Header Standards
We define baseline security controls mandated by NIST and OWASP for web applications:
- **HSTS:** Enforces HTTPS encryption and prevents SSL-stripping.
- **CSP:** Enforces strict execution origins, stopping Cross-Site Scripting (XSS).
- **X-Frame-Options:** Prevents UI redressing (Clickjacking).
- **X-Content-Type-Options:** Prevents MIME-sniffing bypasses.
- **Referrer-Policy:** Blocks confidential token leaks in HTTP Referer headers.

In [2]:
SECURITY_HEADERS = {
    'Strict-Transport-Security': {'severity': 'HIGH', 'desc': 'Enforces HTTPS encryption'},
    'Content-Security-Policy': {'severity': 'HIGH', 'desc': 'Restricts malicious script injection'},
    'X-Frame-Options': {'severity': 'MEDIUM', 'desc': 'Prevents Clickjacking UI redressing'},
    'X-Content-Type-Options': {'severity': 'LOW', 'desc': 'Stops MIME-confusion attacks'},
    'Referrer-Policy': {'severity': 'LOW', 'desc': 'Shields referrer privacy'},
    'Permissions-Policy': {'severity': 'LOW', 'desc': 'Restricts browser hardware sensors'}
}
print(f'[+] Configured {len(SECURITY_HEADERS)} security header benchmarks.')

### 🔍 Step 3: Conducting Live Security Scan on Target
We send probe requests to inspect headers, cookies, and CORS policies.

In [3]:
target_url = 'https://example.com'
response = requests.get(target_url, timeout=10, verify=False)

findings = []
for header, meta in SECURITY_HEADERS.items():
    if header not in response.headers:
        findings.append({'header': header, 'severity': meta['severity'], 'desc': meta['desc']})

print(f'=== AUDIT RESULTS FOR: {target_url} ===')
print(f'Total Missing Security Headers: {len(findings)}')
for f in findings:
    print(f"  [{f['severity']}] Missing {f['header']} - {f['desc']}")

### 🔒 Step 4: SSL/TLS Certificate Handshake & Expiry Validation
Ensures the target's public-key infrastructure has valid trust chains and adequate validity remaining.

In [4]:
parsed = urlparse(target_url)
domain = parsed.netloc
context = ssl.create_default_context()
with socket.create_connection((domain, 443), timeout=5) as sock:
    with context.wrap_socket(sock, server_hostname=domain) as ssock:
        cert = ssock.getpeercert()
        cipher = ssock.cipher()
        print(f"Cipher Suite: {cipher[0]}")
        print(f"TLS Version:  {ssock.version()}")
        print(f"Cert Expiry:  {cert['notAfter']}")

### 📋 Step 5: Report Generation Summary
The tool generates an exportable HTML and JSON security audit report saved in `reports/` with actionable hardening steps.